In [1]:
# Run if necessary
!pip install interpreto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.9/268.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 3.6 MB/s eta 0:00:00


In [3]:
from transformers import AutoModelForCausalLM
from interpreto import ModelWithSplitPoints
from datasets import load_dataset
import torch
from interpreto.concepts.methods.overcomplete import BatchTopKSAEConcepts, DeadNeuronsReanimationLoss
from interpreto.concepts import TopKInputs

# Extracting concepts from an LLM using the `interpreto` package

> This tutorial is heavily inspired from [the interpreto documentation](https://for-sight-ai.github.io/interpreto/).

## Splitting an LLM

We will work with the GPT-2 model for a minimal example of the detailed pipeline. But you can naturally use larger models (with a beefier gpu).

Here we (arbitrarily) split at the tenth layers.

To split the model, we use the [`interpreto.ModelWithSplitPoints`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/) function which wraps around the `transformers` model and allows the computation of activations at the specified `split_points`.


In [4]:
# Load and split the LLM
mwsp = ModelWithSplitPoints(
    "gpt2",
    split_points=[9], # split at the 10th layer
    automodel=AutoModelForCausalLM,
    device_map="cuda",
    batch_size=8,
    dtype=torch.float32,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Gathering a dataset of activations

We will use the [`IMDB`](https://huggingface.co/datasets/stanfordnlp/imdb) dataset to build a dataset of activations.

> ⚠️ **Warning**
>
> The dataset used has a considerable impact on the concept-space obtained. Hence, in practice, we recommend to use a subset of the model training set.
>
> The concept model we train (SAEs or other), find patterns in the dataset of activations, which explains the dependence of the concepts found on the activations dataset.

> 🔥 **Tip**
>
> The larger the dataset the better. But at the cost of computation time to get activations.
>
> Hence, the best dataset for SAEs would be one where each token is only seen once. But this is harder in practice, and such pipeline is not covered in this tutorial.

In [5]:
# We will take the first 4000 reviews
imdb = load_dataset("stanfordnlp/imdb")["train"]["text"][:4000]

# ignore special tokens activations
TOKEN = ModelWithSplitPoints.activation_granularities.TOKEN

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
# compute the activations on the subset of the IMDB dataset
activations_dict = mwsp.get_activations(
    inputs=imdb,
    activation_granularity=TOKEN,
    tqdm_bar=True,
)

Computing activations:   0%|          | 0/500 [00:00<?, ?batch/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Computing activations: 100%|██████████| 500/500 [04:45<00:00,  1.75batch/s]


In [7]:
# it is possible to compute activations for several split points
# hence, we need to extract the activations for the split point we are interested in
activations = mwsp.get_split_activations(activations_dict)
print(f"{activations.shape = }")

activations.shape = torch.Size([1156135, 768])


## Fit a concept model (SAE) on the activations dataset

Now we can fit a concept model on the activations.

In particular, we use the [`interpreto.concepts.BatchTopK`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/sae/#interpreto.concepts.methods.BatchTopKConcepts) function to fit an SAE.

In [8]:
top_k_individual = 10
concept_model_batch_size = 1024
epochs = 80

# instantiate the concept explainer with the splitted model
concept_explainer = BatchTopKSAEConcepts(
    mwsp,
    nb_concepts=800,
    device="cuda",
    top_k=top_k_individual * concept_model_batch_size,
)

In [9]:
# train the SAE on the activations
log = concept_explainer.fit(
    activations=activations.to(torch.float32),
    criterion=DeadNeuronsReanimationLoss,  # set an MSE loss with dead neurons reanimation
    optimizer_class=torch.optim.Adam,
    scheduler_class=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_kwargs={"T_max": epochs, "eta_min": 1e-6},
    lr=1e-3,
    nb_epochs=epochs,
    batch_size=concept_model_batch_size,
    monitoring=1,
)

Epoch[1/80], Loss: 14.1513, R2: 0.6577, L0: 10.2235, Dead Features: 0.0%, Time: 13.2250 seconds
Epoch[2/80], Loss: 11.5284, R2: 0.7212, L0: 10.0922, Dead Features: 30.3%, Time: 12.0734 seconds
Epoch[3/80], Loss: 11.1547, R2: 0.7303, L0: 10.0618, Dead Features: 30.5%, Time: 11.5192 seconds
Epoch[4/80], Loss: 10.8977, R2: 0.7361, L0: 10.0715, Dead Features: 21.1%, Time: 11.9152 seconds
Epoch[5/80], Loss: 10.5279, R2: 0.7451, L0: 10.0953, Dead Features: 10.9%, Time: 11.5288 seconds
Epoch[6/80], Loss: 10.0806, R2: 0.7559, L0: 10.2179, Dead Features: 0.4%, Time: 11.4797 seconds
Epoch[7/80], Loss: 9.6655, R2: 0.7657, L0: 10.2057, Dead Features: 0.6%, Time: 12.0498 seconds
Epoch[8/80], Loss: 9.3826, R2: 0.7727, L0: 10.2235, Dead Features: 0.0%, Time: 11.7958 seconds
Epoch[9/80], Loss: 9.1425, R2: 0.7784, L0: 10.2235, Dead Features: 0.0%, Time: 12.1394 seconds
Epoch[10/80], Loss: 8.9798, R2: 0.7824, L0: 10.2235, Dead Features: 0.9%, Time: 11.9272 seconds
Epoch[11/80], Loss: 8.8671, R2: 0.7852,

## Interpreting the concepts using topK words

Once the SAE has been trained, it is time to interpret its neurons.

Using the [`interpreto.concepts.TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/concepts_interpretations/#interpreto.concepts.interpretations.TopKInputs), it is possible to extract the top-k words that activates the neurons the most.

In [10]:
WORD = ModelWithSplitPoints.activation_granularities.WORD

interpretation_method = TopKInputs(
    concept_explainer=concept_explainer,
    activation_granularity=WORD,
    concept_encoding_batch_size=concept_model_batch_size,
    k=10,  # get the top 10 words for each concepts
    concept_model_device="cuda"
)

In [11]:
#concept_explainer.concept_model.to(torch.bfloat16)

interpretations = interpretation_method.interpret(
    inputs=imdb[:250],  # larger datasets lead to better interpretations
    latent_activations=None,  # we use a different granularity hence we cannot reuse the activations
    concepts_indices="all",  # interpret all concepts
)

for concept_idx, words in list(interpretations.items())[:10]:
    print(f"Concept {concept_idx}: {list(words.keys()) if words else None}")

Concept 0: [' really', ' REALLY', ' Really', 'Really', ',', ' truly', ' Truly', ' actually', ' do', '.']
Concept 1: ['The', ' worst', ' thing', ' only', ' about', ' story', ' is', ' Crush', ' that', ' it']
Concept 2: ['...', ' ...', '...<', '..', '....', '"...', ' and', 'nothing', 'no', 'there']
Concept 3: [' part', 'Part', ' parts', ' Part', ' piece', ' version', ' aspect', ' section', 'part', ' aspects']
Concept 4: [' sense', ' understand', ' understandable', ' misinterpreted', ' understood', ' accurate', ' in', ' grasped', ' discernable', 'sense']
Concept 5: [' Ed', ' Wood', ' Edward', ' dub', ' led', ' Brad', ' Frank', ' seductive', "'s", ' Edgerton']
Concept 6: [' was', ' were', ' wasn', ' didn', ' had', ' obviously', ' Did', "'t", ' not', ' Was']
Concept 7: ['Okay', ' first', ',', ' of', ' I', "'t", ' all', ' didn', ' to', ' watch']
Concept 8: [' The', 'The', ' the', 'the', ' His', ' entire', ' whole', ' Her', ' Their', ' his']
Concept 9: [' even', ' again', ' eventually', ' also